In [ ]:
# Cell 1
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score

sns.set_style("whitegrid")
print("Libraries loaded")

In [ ]:
# Cell 2
DATA_PATH = "../data/raw/dataset.csv"   # change to your file name
PROCESSED_PATH = "../data/processed/cleaned_dataset.csv"

print("Working dir:", os.getcwd())
print("Data path:", DATA_PATH)

In [ ]:
# Cell 3
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
# Cell 4
df.info()
df.describe(include="all").T

In [ ]:
# Cell 5
df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# Cell 6
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(exclude=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining nulls:", df.isnull().sum().sum())

In [ ]:
# Cell 7
num_cols = df.select_dtypes(include=[np.number]).columns[:6]
df[num_cols].hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8
plt.figure(figsize=(10, 6))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Cell 9
df_model = pd.get_dummies(df, drop_first=True)

TARGET = "target"   # <-- change this to your target column
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

if y.nunique() <= 10 and y.dtype != float:
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, preds))
else:
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print("RMSE:", mean_squared_error(y_test, preds) ** 0.5)

In [ ]:
# Cell 10
os.makedirs("../data/processed", exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print("Saved cleaned file:", PROCESSED_PATH)